# 02 — Knowledge Graph construction

This notebook turns the cleaned MSD × Lakh × jSymbolic dataframe into RDF triples
that **extend** the project ontology in `notebooks/MusicRecSyst.ttl`.

The original ontology file is **never modified** — every triple we mint is
written to a sibling file, `notebooks/MusicRecSyst_populated.ttl`, which is a
fresh copy of the base ontology each time `KGBuilder(...)` is instantiated with
`overwrite_copy=True`.

Pipeline:

1. **Load** the per-track parquet (`data/processed/lakh_msd_dataset.parquet`).
2. **Project** to the columns we care about for the KG (`select_kg_columns`).
3. **Join** on the MIDI md5 with the jSymbolic features in
   `data/interim/interim.csv` (`load_interim_features` + `merge_parquet_with_interim`).
4. **Derive** a categorical `tempo_class` column from `Mean_Tempo` using the
   *Music Theory Academy* scheme (Larghissimo … Prestissimo).
5. **Populate** the ontology through `KGBuilder`, which mints individuals for
   artists, tracks, performances, genres, instruments, and a fixed vocabulary
   of `mrc:TempoClass` instances.
6. **Serialize** to Turtle and inspect a few triples.

All helpers live in `src/data/kg/`.

In [6]:
"""Bootstrap: make ``src`` importable and pin the project root."""
from __future__ import annotations

import sys
from pathlib import Path

# notebooks/ → project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR          = PROJECT_ROOT / "data"
PARQUET_PATH      = DATA_DIR / "processed" / "lakh_msd_dataset.parquet"
INTERIM_CSV       = DATA_DIR / "interim"   / "interim.csv"
KG_INPUT_PQ       = DATA_DIR / "interim"   / "kg_input.parquet"          # cached, cleaned join
TASTE_FILTERED_PQ = DATA_DIR / "processed" / "taste_profile_filtered.parquet"  # full filtered profile (≥5 plays)
KG_TASTE_PQ       = DATA_DIR / "interim"   / "kg_taste_profile.parquet"  # restricted to KG songs, re-filtered
ONTO_BASE         = PROJECT_ROOT / "notebooks" / "MusicRecSyst.ttl"
ONTO_OUT          = PROJECT_ROOT / "notebooks" / "MusicRecSyst_populated.ttl"

# Set to True to ignore the cached parquet / TTL and rebuild from scratch.
FORCE_REBUILD = False

print("PROJECT_ROOT       :", PROJECT_ROOT)
print("parquet            :", PARQUET_PATH.exists(),      PARQUET_PATH)
print("interim csv        :", INTERIM_CSV.exists(),       INTERIM_CSV)
print("kg input pq        :", KG_INPUT_PQ.exists(),       KG_INPUT_PQ)
print("taste filtered pq  :", TASTE_FILTERED_PQ.exists(), TASTE_FILTERED_PQ)
print("kg taste pq        :", KG_TASTE_PQ.exists(),       KG_TASTE_PQ)
print("ontology base      :", ONTO_BASE.exists(),         ONTO_BASE)


PROJECT_ROOT       : /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project
parquet            : True /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/processed/lakh_msd_dataset.parquet
interim csv        : True /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/interim.csv
kg input pq        : False /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/kg_input.parquet
taste filtered pq  : True /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/processed/taste_profile_filtered.parquet
kg taste pq        : False /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/kg_taste_profile.parquet
ontology base      : True /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/notebooks/MusicRecSyst.ttl


In [32]:
# Force-reload the data.kg subpackage so notebook iterations pick up code edits
# without needing a full kernel restart.
import importlib, sys
for _name in [m for m in sys.modules if m == "data.kg" or m.startswith("data.kg.")]:
    importlib.reload(sys.modules[_name])


## 1 · Build (or load) the cleaned KG input

We want a single, reproducible dataframe that drives the KG build. The pipeline is:

1. Load the per-track parquet, project to `DEFAULT_KG_COLUMNS`.
2. Load `interim.csv` (jSymbolic features) and join on the **MIDI md5**
   (extracted from the Windows-style path stem).
3. **Drop** every row that has no jSymbolic features attached — only ~37 % of
   the parquet rows have a matching jSymbolic extraction (`7,113 / 19,137`),
   and the KG entries for the rest would be strictly poorer than what we
   already store on disk. We choose to keep *only* tracks that have both
   MSD metadata **and** jSymbolic features, so every node in the graph carries
   the full feature stack.
4. Apply the rename map (`KG_RENAME_MAP`): `release → album_name`,
   `key_name → key`, `mode_name → mode`. The integer-coded `key`/`mode`
   columns from the MSD parquet are *not* selected — we keep only the
   human-readable spellings.
5. Derive the categorical `tempo_class` from `Mean_Tempo` (Music Theory Academy
   nine-class scheme).
6. **Cache** the result to `data/interim/kg_input.parquet`.

On subsequent runs (when `FORCE_REBUILD=False` and the cache exists) we just
load the parquet — no re-joining, no re-cleaning.

Columns kept from the parquet (`DEFAULT_KG_COLUMNS` in
`src/data/kg/variable_selection.py`), shown under their **post-rename** names:

* identifiers — `track_id`, `midi_path`, `song_id`, `artist_id`, `artist_mbid`
* labels      — `artist_name`, `title`, `album_name` *(was `release`)*, `year`
* tonality    — `key` *(was `key_name`)*, `key_confidence`,
                `mode` *(was `mode_name`)*, `mode_confidence`
* MSD audio   — `time_signature`, `time_signature_confidence`, `loudness`
* tags        — `primary_genre`, `top3_genres`, `artist_terms`
* MIDI        — `midi_n_instruments`, `midi_instrument_names`

On top of these we attach all `INTERIM_KG_FEATURES` (jSymbolic numerics:
`Mean_Tempo`, `Number_of_Pitches`, `Pitch_Variety`, etc.) via the inner-join,
and finally the derived `tempo_class` column.


In [14]:
import pandas as pd

from data.kg import (
    DEFAULT_KG_COLUMNS,
    INTERIM_KG_FEATURES,
    select_kg_columns,
    load_interim_features,
    merge_parquet_with_interim,
)
from data.kg.tempo_classes import add_tempo_class_column


def build_kg_input() -> pd.DataFrame:
    """Project parquet → join with interim → drop unmatched → add tempo_class."""
    raw   = pd.read_parquet(PARQUET_PATH)
    print(f"  raw parquet           : {raw.shape}")

    kg_df = select_kg_columns(raw)
    print(f"  projected to KG cols  : {kg_df.shape}")

    interim_df = load_interim_features(INTERIM_CSV, INTERIM_KG_FEATURES)
    print(f"  interim features      : {interim_df.shape}")

    # inner-join: keep only tracks that have BOTH MSD metadata AND jSymbolic
    merged = merge_parquet_with_interim(
        kg_df, interim_df, midi_path_col="midi_path", how="inner",
    )
    print(f"  inner-joined          : {merged.shape}")

    # belt-and-braces: drop any row that ended up without a Mean_Tempo
    before = len(merged)
    merged = merged.dropna(subset=["Mean_Tempo"]).reset_index(drop=True)
    if len(merged) != before:
        print(f"  dropped {before - len(merged):,} rows w/o Mean_Tempo")

    merged = add_tempo_class_column(merged, tempo_col="Mean_Tempo",
                                    out_col="tempo_class")
    return merged


if KG_INPUT_PQ.exists() and not FORCE_REBUILD:
    print(f"loading cached KG input from {KG_INPUT_PQ}")
    merged = pd.read_parquet(KG_INPUT_PQ)
    # tempo_class round-trips as plain object/category — rebuild the ordered Categorical
    merged = add_tempo_class_column(merged, tempo_col="Mean_Tempo",
                                    out_col="tempo_class")
else:
    print("building KG input from scratch …")
    merged = build_kg_input()
    KG_INPUT_PQ.parent.mkdir(parents=True, exist_ok=True)
    # Categorical doesn't survive parquet round-trip cleanly with dtype=category
    # in some pyarrow versions, so cast to plain string before writing.
    out = merged.copy()
    out["tempo_class"] = out["tempo_class"].astype("string")
    out.to_parquet(KG_INPUT_PQ, index=False)
    print(f"  cached → {KG_INPUT_PQ} ({KG_INPUT_PQ.stat().st_size/1024:,.1f} KiB)")

print(f"\nfinal KG input: {merged.shape}")
merged.head(3)


building KG input from scratch …
  raw parquet           : (19037, 40)
  projected to KG cols  : (19037, 21)
Loaded interim features: 7113 rows × 31 cols.
  interim features      : (7113, 31)
Merged: 7,113 rows total, 7,113 with interim features (100.0%).
  inner-joined          : (7113, 53)
Loaded interim features: 7113 rows × 31 cols.
  interim features      : (7113, 31)
Merged: 7,113 rows total, 7,113 with interim features (100.0%).
  inner-joined          : (7113, 53)
  cached → /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/kg_input.parquet (2,515.0 KiB)

final KG input: (7113, 54)
  cached → /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/kg_input.parquet (2,515.0 KiB)

final KG input: (7113, 54)


,track_id,midi_path,song_id,artist_id,artist_mbid,artist_name,title,album_name,key,key_confidence,...,Acoustic_Guitar_Prevalence,Electric_Guitar_Prevalence,Brass_Prevalence,Woodwinds_Prevalence,Average_Number_of_Independent_Voices,Variability_of_Number_of_Independent_Voices,Voice_Equality_-_Number_of_Notes,Variation_of_Dynamics,Average_Note_to_Note_Change_in_Dynamics,tempo_class
0,TRAAAGR128F425B14B,/home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJ...,SONRWUU12AF72A4283,ARGE7G11187FB37E05,7bd9e20e-74b9-446a-a2ed-a223f82a36e7,Cyndi Lauper,Into The Nightlife,Bring Ya To The Brink,A,0.608,...,0.0,0.12870,0.01361,0.01717,3.766,2.152,557.70,26.410,13.070,Allegro
1,TRAABVM128F92CA9DC,/home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJ...,SOXLBJT12A8C140925,ARYKCQI1187FB3B18F,eeacb319-8d4c-48e0-80a0-944e71c375bf,Tesla,Caught In A Dream,Gold,G,0.725,...,0.0,0.17370,0.00000,0.00000,4.001,1.562,418.80,19.690,13.590,Adagio
2,TRAADKW128E079503A,/home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJ...,SOJCRUY12A67ADA4C2,ARAO91X1187B98CCA4,1129817c-488a-4096-80c1-77fc1b107c93,Tracy Chapman,Fast Car (LP Version),Tracy Chapman,D,0.018,...,0.3,0.08182,0.00000,0.22610,5.859,1.658,65.25,9.977,3.734,Andante


## 2 · `tempo_class` reference + distribution

We use the [Music Theory Academy](https://www.musictheoryacademy.com/understanding-music/tempo/)
nine-class scheme.  Each class is a half-open BPM interval `[lo, hi)`; the top
class extends to `+∞`.

| name          | range (BPM) |
|---------------|-------------|
| Larghissimo   | < 20        |
| Grave         | 20 – 40     |
| Largo         | 40 – 60     |
| Adagio        | 60 – 76     |
| Andante       | 76 – 108    |
| Moderato      | 108 – 120   |
| Allegro       | 120 – 168   |
| Presto        | 168 – 200   |
| Prestissimo   | ≥ 200       |

The classifier reads from `Mean_Tempo` (jSymbolic, full per-MIDI estimate).
After the inner-join above, every row has a non-null `Mean_Tempo`, so every row
also has a `tempo_class`.

In [10]:
from data.kg.tempo_classes import tempo_class_table

print(f"tempo_class distribution (n = {len(merged):,}):")
print(merged["tempo_class"].value_counts(dropna=False).sort_index())

tempo_class_table()


tempo_class distribution (n = 7,113):
tempo_class
Larghissimo       0
Grave            15
Largo           109
Adagio          853
Andante        2322
Moderato        844
Allegro        2611
Presto          279
Prestissimo      80
Name: count, dtype: int64


,tempo_class,min_bpm,max_bpm,description
0,Larghissimo,0.0,20.0,Extremely slow (< 20 BPM)
1,Grave,20.0,40.0,Slow and solemn (20–40 BPM)
2,Largo,40.0,60.0,Very slow (40–60 BPM)
3,Adagio,60.0,76.0,Slow and stately (60–76 BPM)
4,Andante,76.0,108.0,Walking pace (76–108 BPM)
5,Moderato,108.0,120.0,Moderately (108–120 BPM)
6,Allegro,120.0,168.0,Fast (120–168 BPM)
7,Presto,168.0,200.0,Very fast (168–200 BPM)
8,Prestissimo,200.0,inf,Very very fast (≥ 200 BPM)


## 3 · Restrict the taste profile to KG songs

`taste_profile_filtered.parquet` (built in `00b_user_data_integration.ipynb`)
contains every `(user_id, song_id, play_count)` interaction surviving the
`MIN_PLAYS_PER_USER = 5` cold-start filter, **at the time it was built**.

But the KG only covers the 7,113 tracks that have *both* MSD metadata and
jSymbolic features — so any song outside that set effectively disappears from
our recommendation universe. Once we drop those interactions some users may
fall below the 5-play threshold, so we **re-apply the filter** to keep the
distribution honest.

The helper `load_or_build_kg_taste_profile`:

1. loads `taste_profile_filtered.parquet`,
2. keeps only rows whose `song_id` is in the KG (`merged["song_id"]`),
3. re-applies `min_plays_per_user=5`,
4. caches the result to `data/interim/kg_taste_profile.parquet`.

Subsequent runs just reload the cache (unless `FORCE_REBUILD=True`).


In [11]:
from data.kg import load_or_build_kg_taste_profile

kg_song_ids = set(merged["song_id"].dropna().unique())
print(f"KG covers {len(kg_song_ids):,} unique song_ids")

taste = load_or_build_kg_taste_profile(
    cache_path=KG_TASTE_PQ,
    taste_source=TASTE_FILTERED_PQ,
    kg_song_ids=kg_song_ids,
    min_plays_per_user=5,
    force_rebuild=FORCE_REBUILD,
)

print(f"\nKG-restricted taste profile: {taste.shape}")
print(f"  unique users : {taste['user_id'].nunique():,}")
print(f"  unique songs : {taste['song_id'].nunique():,}")
print(f"  interactions : {len(taste):,}")
print("\nplays per user (post-filter):")
print(taste.groupby("user_id").size().describe())


KG covers 7,013 unique song_ids
[user_data] building cache from /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/processed/taste_profile_filtered.parquet
[user_data] KG-song restriction: 3,048,959 → 2,908,088 rows  (−140,871)  |  users 299,156 → 299,156  |  songs 7,227 → 7,013
[user_data] KG-song restriction: 3,048,959 → 2,908,088 rows  (−140,871)  |  users 299,156 → 299,156  |  songs 7,227 → 7,013
[user_data] cold-start (min=5): 2,908,088 → 2,853,630 rows  (−54,458, −14,119 users)
[user_data] Final:  2,853,630 interactions  |  285,037 users  |  7,010 songs
[user_data] cold-start (min=5): 2,908,088 → 2,853,630 rows  (−54,458, −14,119 users)
[user_data] Final:  2,853,630 interactions  |  285,037 users  |  7,010 songs
[user_data] cached → /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/kg_taste_profile.parquet (22,993.7 KiB)

KG-restricted taste profile: (2853630, 3)
  unique users : 285,037
  unique songs : 7,010
  interactions : 2,853,63

## 4 · Populate the ontology

`KGBuilder` opens a **fresh copy** of `MusicRecSyst.ttl` at
`MusicRecSyst_populated.ttl` and adds new triples on top of it.  The base file
is never modified.

For each row it mints (idempotently, by URI):

* `mo:MusicArtist` — keyed by `artist_id` / `artist_mbid`
* `mrc:MSDTrack` — keyed by `track_id`, with `dcterms:title`, `dcterms:date`,
  release info
* `mo:Performance` — links the artist to the track, carries `mo:tempo`,
  `mo:key`, `mrc:hasTempoClass`, and one `mo:instrument` per GM instrument name
* `mrc:Genre` — one per unique genre/tag label, attached to the artist
* `mo:Instrument` — one per unique GM name
* `mrc:*` datatype properties for the jSymbolic numeric features

Tempo-class individuals (Larghissimo … Prestissimo) are declared once with
`mrc:minBPM` / `mrc:maxBPM` so SPARQL queries can range-filter by class.

If `MusicRecSyst_populated.ttl` already exists and `FORCE_REBUILD` is `False`
we just load it back; otherwise we rebuild from scratch and overwrite it.


In [33]:
from data.kg import KGBuilder

if ONTO_OUT.exists() and not FORCE_REBUILD:
    print(f"loading cached populated graph from {ONTO_OUT}")
    # Re-instantiating with overwrite_copy=False reuses the existing TTL
    builder = KGBuilder(
        base_ttl=ONTO_BASE,
        out_ttl=ONTO_OUT,
        overwrite_copy=False,
    )
    print("graph stats:", builder.stats())
else:
    print("building knowledge graph from scratch …")
    builder = KGBuilder(
        base_ttl=ONTO_BASE,
        out_ttl=ONTO_OUT,
        overwrite_copy=True,
    )
    builder.add_tempo_class_individuals()

    counts = builder.populate_from_dataframe(merged, max_rows=None, verbose=True)
    print("\nminted (this run):", counts)
    print("graph stats      :", builder.stats())

    builder.save()
    print(f"\nwrote {ONTO_OUT}  ({ONTO_OUT.stat().st_size / 1024:,.1f} KiB)")


building knowledge graph from scratch …
KG population summary: {'artists': 3668, 'tracks': 7013, 'performances': 7013, 'genres': 645, 'instruments': 129, 'rows_skipped': 0}

minted (this run): {'artists': 3668, 'tracks': 7013, 'performances': 7013, 'genres': 645, 'instruments': 129, 'rows_skipped': 0}
graph stats      : {'triples': 259412, 'artists': 3668, 'tracks': 7013, 'performances': 7013, 'genres': 645, 'instruments': 129, 'tempo_classes': 9}
KG population summary: {'artists': 3668, 'tracks': 7013, 'performances': 7013, 'genres': 645, 'instruments': 129, 'rows_skipped': 0}

minted (this run): {'artists': 3668, 'tracks': 7013, 'performances': 7013, 'genres': 645, 'instruments': 129, 'rows_skipped': 0}
graph stats      : {'triples': 259412, 'artists': 3668, 'tracks': 7013, 'performances': 7013, 'genres': 645, 'instruments': 129, 'tempo_classes': 9}

wrote /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/notebooks/MusicRecSyst_populated.ttl  (13,953.4 KiB)

wrote /home

## 5 · Inspect

A couple of SPARQL sanity checks against the in-memory graph.


In [19]:
q_tracks_per_class = """
PREFIX mrc: <http://purl.org/ontology/mrc/>
PREFIX mo:  <http://purl.org/ontology/mo/>
SELECT ?className (COUNT(DISTINCT ?perf) AS ?n)
WHERE {
    ?perf a mo:Performance ;
          mrc:hasTempoClass ?tc .
    ?tc <http://www.w3.org/2000/01/rdf-schema#label> ?className .
}
GROUP BY ?className
ORDER BY DESC(?n)
"""
print("performances per tempo class:")
for row in builder.g.query(q_tracks_per_class):
    print(f"  {str(row[0]):<14} {int(row[1]):>5}")

q_top_genres = """
PREFIX mrc: <http://purl.org/ontology/mrc/>
SELECT ?label (COUNT(?a) AS ?n)
WHERE {
    ?a mrc:hasGenre ?g .
    ?g <http://www.w3.org/2000/01/rdf-schema#label> ?label .
}
GROUP BY ?label
ORDER BY DESC(?n)
LIMIT 10
"""
print("\ntop 10 genres by artist count:")
for row in builder.g.query(q_top_genres):
    print(f"  {str(row[0]):<30} {int(row[1]):>5}")


performances per tempo class:
  Allegro         2573
  Andante         2293
  Adagio           840
  Moderato         834
  Presto           273
  Largo            108
  Prestissimo       79
  Grave             13

top 10 genres by artist count:
  rock                             960
  electronic                       217
  hip hop                          150
  trance                           108
  jazz                              99
  pop rock                          91
  pop                               79
  indie rock                        74
  disco                             67
  singer-songwriter                 63
  Allegro         2573
  Andante         2293
  Adagio           840
  Moderato         834
  Presto           273
  Largo            108
  Prestissimo       79
  Grave             13

top 10 genres by artist count:
  rock                             960
  electronic                       217
  hip hop                          150
  trance                       

## 6 · Wikidata enrichment (instruments + genres)

Local instrument and genre nodes are minted from raw labels (e.g.
`"Acoustic Grand Piano"`, `"new wave"`) and start out as flat leaves with a
single `rdfs:label`. We can do much better by linking them to **Wikidata**:

* `mo:Instrument` labels → Wikidata QIDs that are subclasses of
  `Q34379` *musical instrument*.
* `mrc:Genre` labels → Wikidata QIDs that are subclasses of
  `Q188451` *music genre*.

For each resolved QID we also pull the `wdt:P279*` *subclass-of* chain up to
the type root and mint `rdfs:subClassOf` edges between QID nodes. That gives
SPARQL queries a real taxonomy to walk (e.g.
`?g rdfs:subClassOf* wd:Q11399  # rock`).

Both resolution and chain expansion hit the public Wikidata APIs, so they're
**cached** to JSON in `data/interim/`:

* `wikidata_instruments.json` / `wikidata_instrument_chains.json`
* `wikidata_genres.json` / `wikidata_genre_chains.json`

Subsequent runs are O(1) — we only query labels we haven't seen before.
Network failures are recorded as `null` in the cache so we don't retry them.

> **Note** — the first run takes a few minutes (one HTTP round-trip per
> unique label). Toggle `RUN_WIKIDATA = False` below to skip this section if
> you're iterating offline.


In [34]:
RUN_WIKIDATA = True   # set False to skip the network-bound enrichment

if RUN_WIKIDATA:
    from data.kg import (
        INSTRUMENT_ROOT, GENRE_ROOT,
        resolve_labels, fetch_subclass_chains, enrich_graph_with_wikidata,
    )
    from data.kg.kg_builder import _iter_strings

    WD_INSTR_PQ        = DATA_DIR / "interim" / "wikidata_instruments.json"
    WD_INSTR_CHAINS_PQ = DATA_DIR / "interim" / "wikidata_instrument_chains.json"
    WD_GENRE_PQ        = DATA_DIR / "interim" / "wikidata_genres.json"
    WD_GENRE_CHAINS_PQ = DATA_DIR / "interim" / "wikidata_genre_chains.json"

    # ── 6.1 collect unique labels actually used in the KG ──────────────────
    instr_labels: set[str] = set()
    for cell in merged["midi_instrument_names"]:
        instr_labels.update(_iter_strings(cell))

    genre_labels: set[str] = set()
    for cell in merged["primary_genre"]:
        if isinstance(cell, str) and cell.strip():
            genre_labels.add(cell.strip())
    for col in ("top3_genres", "artist_terms"):
        for cell in merged[col]:
            genre_labels.update(_iter_strings(cell))

    print(f"unique instrument labels : {len(instr_labels):,}")
    print(f"unique genre labels      : {len(genre_labels):,}")

    # ── 6.2 resolve labels → QIDs (cached) ─────────────────────────────────
    instr_map = resolve_labels(
        sorted(instr_labels), INSTRUMENT_ROOT, cache_path=WD_INSTR_PQ,
        force_refresh=FORCE_REBUILD,
    )
    genre_map = resolve_labels(
        sorted(genre_labels), GENRE_ROOT, cache_path=WD_GENRE_PQ,
        force_refresh=FORCE_REBUILD,
    )
    print(f"\ninstrument hits: {sum(1 for v in instr_map.values() if v):,}"
          f" / {len(instr_map):,}")
    print(f"genre hits     : {sum(1 for v in genre_map.values() if v):,}"
          f" / {len(genre_map):,}")


unique instrument labels : 129
unique genre labels      : 645
[wikidata] loaded 129 cached label→QID entries from wikidata_instruments.json
[wikidata] resolving 0 new labels (type_root=Q110295396) …
[wikidata] cache → /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/wikidata_instruments.json
[wikidata] loaded 645 cached label→QID entries from wikidata_genres.json
[wikidata] resolving 0 new labels (type_root=Q188451) …
[wikidata] cache → /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/wikidata_genres.json

instrument hits: 67 / 129
genre hits     : 507 / 645


In [35]:
if RUN_WIKIDATA:
    # ── 6.3 expand subclass-of chains for the resolved QIDs (cached) ───────
    instr_qids = [q for q in instr_map.values() if q]
    genre_qids = [q for q in genre_map.values() if q]

    instr_chains = fetch_subclass_chains(
        instr_qids, INSTRUMENT_ROOT, cache_path=WD_INSTR_CHAINS_PQ,
        force_refresh=FORCE_REBUILD,
    )
    genre_chains = fetch_subclass_chains(
        genre_qids, GENRE_ROOT, cache_path=WD_GENRE_CHAINS_PQ,
        force_refresh=FORCE_REBUILD,
    )

    # ── 6.4 fold into the populated KG ─────────────────────────────────────
    enrich_counts = enrich_graph_with_wikidata(
        builder,
        instrument_map=instr_map,
        genre_map=genre_map,
        instrument_chains=instr_chains,
        genre_chains=genre_chains,
        add_hierarchy=True,
    )
    print("\nenrichment summary :", enrich_counts)
    print("graph stats (after):", builder.stats())

    builder.save()
    print(f"\nupdated {ONTO_OUT}  ({ONTO_OUT.stat().st_size / 1024:,.1f} KiB)")


[wikidata] loaded 46 cached subclass chains from wikidata_instrument_chains.json
[wikidata] expanding 0 new QIDs (root=Q110295396) …
[wikidata] cache → /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/wikidata_instrument_chains.json
[wikidata] loaded 442 cached subclass chains from wikidata_genre_chains.json
[wikidata] expanding 0 new QIDs (root=Q188451) …
[wikidata] cache → /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/data/interim/wikidata_genre_chains.json
[wikidata] enrichment summary: {'instrument_links': 67, 'genre_links': 507, 'qid_nodes': 654, 'subclass_edges': 1061}

enrichment summary : {'instrument_links': 67, 'genre_links': 507, 'qid_nodes': 654, 'subclass_edges': 1061}
graph stats (after): {'triples': 261852, 'artists': 3668, 'tracks': 7013, 'performances': 7013, 'genres': 1039, 'instruments': 170, 'tempo_classes': 9}

updated /home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/DL-KG-project/notebooks/MusicRecSyst_populated.ttl  

### 6.5 · Sanity-check the Wikidata hierarchy

Two SPARQL probes against the in-memory graph:

1. how many local genre/instrument leaves got an `owl:sameAs` to a Wikidata QID;
2. for a single example (`piano`), walk `rdfs:subClassOf*` from the local
   instrument node up through the Wikidata chain — should give us
   *piano → struck-string instrument → string instrument → musical instrument*
   (whatever `wdt:P279*` returned, bounded to the music domain).


In [36]:
if RUN_WIKIDATA:
    q_link_counts = """
    PREFIX owl:  <http://www.w3.org/2002/07/owl#>
    PREFIX mo:   <http://purl.org/ontology/mo/>
    PREFIX mrc:  <http://purl.org/ontology/mrc/>
    SELECT
        (COUNT(DISTINCT ?gi) AS ?genres_linked)
        (COUNT(DISTINCT ?ii) AS ?instruments_linked)
    WHERE {
      OPTIONAL { ?gi a mrc:Genre      ; owl:sameAs ?gw . FILTER(STRSTARTS(STR(?gw), "http://www.wikidata.org/")) }
      OPTIONAL { ?ii a mo:Instrument  ; owl:sameAs ?iw . FILTER(STRSTARTS(STR(?iw), "http://www.wikidata.org/")) }
    }
    """
    for row in builder.g.query(q_link_counts):
        print(f"genres linked      : {int(row.genres_linked):>4}")
        print(f"instruments linked : {int(row.instruments_linked):>4}")

    # Walk piano-family hierarchy: local node → owl:sameAs → wd → rdfs:subClassOf*
    q_piano_chain = """
    PREFIX owl:  <http://www.w3.org/2002/07/owl#>
    PREFIX mo:   <http://purl.org/ontology/mo/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?local_label ?ancestor ?ancestor_label
    WHERE {
      ?local a mo:Instrument ;
             rdfs:label ?local_label ;
             owl:sameAs ?wd .
      FILTER(CONTAINS(LCASE(STR(?local_label)), "piano"))
      ?wd rdfs:subClassOf* ?ancestor .
      OPTIONAL { ?ancestor rdfs:label ?ancestor_label . FILTER(LANG(?ancestor_label) = "en") }
    }
    ORDER BY ?local_label ?ancestor
    """
    print("\npiano-family instruments → Wikidata ancestors :")
    last_local = None
    for row in builder.g.query(q_piano_chain):
        local = str(row.local_label)
        if local != last_local:
            print(f"\n  {local}")
            last_local = local
        qid = str(row.ancestor).rsplit("/", 1)[-1]
        print(f"     {qid:10s} {row.ancestor_label}")


genres linked      :  507
instruments linked :   67

piano-family instruments → Wikidata ancestors :

  Acoustic Grand Piano
     Q34379     musical instrument
     Q52954     keyboard instrument
     Q5994      piano

  Bright Acoustic Piano
     Q34379     musical instrument
     Q52954     keyboard instrument
     Q5994      piano

  Electric Grand Piano
     Q34379     musical instrument
     Q52954     keyboard instrument
     Q5994      piano

  Honky-tonk Piano
     Q34379     musical instrument
     Q52954     keyboard instrument
     Q5994      piano
